In [0]:
df = spark.read.option("header", "true").option("inferSchema","true").csv("/Volumes/workspace/bronze/bronze/Grocery_Inventory_and_Sales_Dataset.csv")
display(df)

In [0]:
from pyspark.sql.functions import col
 
if spark.catalog.tableExists("bronze.grocery_raw_new"):
    max_ingested = spark.sql("SELECT MAX(inserted_date) as max_ingested FROM bronze.grocery_raw_new").collect()[0]['max_ingested']
    if max_ingested is not None:
        df_new = df.filter(col("inserted_date") > max_ingested)
        new_count = df_new.count()
        if new_count > 0:
            print(f"Ingesting {new_count} new records with ingested_date > {max_ingested}")
            df_new.write.format("delta").mode("append").saveAsTable("bronze.grocery_raw_new")
            print("Ingestion completed.")
        else:
            print("No new records to ingest based on ingested_date.")
    else:
        print("No records in target table. Ingesting all source records.")
        df.write.format("delta").mode("append").saveAsTable("bronze.grocery_raw_new")
        print(f"Ingestion completed - {df.count()} records loaded")
else:
    print("Bronze table doesn't exist, creating with initial load")
    df.write.format("delta").mode("overwrite").saveAsTable("bronze.grocery_raw_new")
    print(f"Initial load completed - {df.count()} records loaded")

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("bronze.grocery_raw_new")

In [0]:
%sql
select * from bronze.grocery_raw_new